# Project Agonistes — Empirical Research Lab (01)

Cell-by-cell, zero-black-box research protocol across 6 stages. Run **smoke** (`RUN_MODE=smoke`) for a fast end-to-end sanity pass, or **full** (`RUN_MODE=full`) for the complete benchmark on the GPU pod.

In [1]:

import os, sys, json, time
from pathlib import Path

# Anchor to the project root (strategy_builder/ + core/ + data/) regardless of
# the kernel's cwd (Jupyter sets kernel cwd to the notebook's directory).
_cwd = Path(os.getcwd())
PROJECT_ROOT = _cwd
for _p in [PROJECT_ROOT, *_cwd.parents]:
    if (_p / "strategy_builder").is_dir() and (_p / "core").is_dir():
        PROJECT_ROOT = _p
        break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"[lab] project root: {PROJECT_ROOT}")

import numpy as np
import pandas as pd

RUN_MODE = os.environ.get("RUN_MODE", "full").strip().lower()
print(f"[lab] RUN_MODE = {RUN_MODE}")
import torch
print(f"[lab] device  = cuda available -> {torch.cuda.is_available()}")


[lab] project root: C:\Users\Priyatanshu Ghosh\Documents\Python Practice\CFA Practice


[lab] RUN_MODE = smoke


[lab] device  = cuda available -> False


In [2]:

# ---------------------------------------------------------------------------
# Config: read from data/benchmark/lab_config.json (survives kernel env
# inheritance) with env-var fallback. One knob for smoke vs full-strength.
# ---------------------------------------------------------------------------
import json
from pathlib import Path
from strategy_builder.trainer import default_device

# Resolve project root regardless of kernel cwd (Jupyter sets it to the
# notebook's directory). Independent of the PRELUDE cell.
_p_root = Path(os.getcwd())
for _p in [_p_root, *_p_root.parents]:
    if (_p / "strategy_builder").is_dir() and (_p / "core").is_dir():
        _p_root = _p
        break

_cfg = {}
_cfg_path = Path(os.environ.get("AG_CONFIG", str(_p_root / "data/benchmark/lab_config.json")))
if _cfg_path.exists():
    try:
        _cfg = json.loads(_cfg_path.read_text())
    except Exception as _e:  # noqa: BLE001
        print(f"[lab] warning: could not read {_cfg_path}: {_e}")
RUN_MODE = _cfg.get("run_mode", os.environ.get("RUN_MODE", "full")).strip().lower()

def _env_int(name, default):
    try:
        return int(_cfg.get(name, os.environ.get(name, str(default))))
    except (TypeError, ValueError):
        return default

def _env_models(name, default):
    raw = _cfg.get(name, os.environ.get(name, ""))
    if isinstance(raw, str):
        raw = raw.strip()
        return [m.strip() for m in raw.split(",") if m.strip()] or default
    if isinstance(raw, list):
        return [str(m).strip() for m in raw if str(m).strip()] or default
    return default

DEVICE = default_device()

# Universe: balanced 29-asset cross-asset panel (US/India/ETF/FX/rates/commo)
UNIVERSE = [
    # US large caps
    "AAPL","AMZN","GOOGL","JPM","META","MSFT","NVDA","TSLA","UNH","XOM",
    # Indian bluechips
    "HDFCBANK.NS","INFY.NS","RELIANCE.NS","SBIN.NS","TCS.NS",
    # Liquid ETFs / indices
    "SPY","QQQ","IWM","EEM","GLD","TLT","^NSEI",
    # FX / rates / commodities
    "EURUSD=X","GBPUSD=X","USDJPY=X","USDINR=X","DX-Y.NYB","^TNX","GC=F",
]

# Walk-forward geometry
TRAIN_MONTHS = 24 if RUN_MODE == "smoke" else 36
TEST_MONTHS  = 4  if RUN_MODE == "smoke" else 6
LOOKBACK     = 32 if RUN_MODE == "smoke" else 64
HIDDEN       = 16 if RUN_MODE == "smoke" else 32
EPOCHS       = 15 if RUN_MODE == "smoke" else 60
SEEDS        = 2  if RUN_MODE == "smoke" else 3
TOP_SEEDS    = 1  if RUN_MODE == "smoke" else 2
BATCH_SIZE   = 256
LR           = 1e-3
SIGMA_TGT    = 0.10
PATIENCE     = 10

# Model rosters (each overridable independently via config/env)
NEURAL_MODELS     = _env_models("neural", ["vlstm", "tft", "nlinear", "lstm", "patchtst"])
CLASSICAL_MODELS  = _env_models("classical", ["ridge", "elasticnet", "xgboost", "lightgbm", "catboost", "knn"])
VOLATILITY_MODELS = _env_models("volatility", ["garch11", "egarch", "gjr_garch", "har_rv"])
BENCH_DIR = Path(_cfg.get("bench_dir", os.environ.get("AG_BENCH_DIR", str(_p_root / "data/benchmark"))))
BENCH_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_MODE:", RUN_MODE, "| Train months:", TRAIN_MONTHS,
      "| Test months:", TEST_MONTHS, "| Lookback:", LOOKBACK,
      "| Epochs:", EPOCHS)


RUN_MODE: smoke | Train months: 24 | Test months: 4 | Lookback: 32 | Epochs: 15


# Stage 1 — Data Panel Extraction & Verification

Connect to the local SQLite `data/agonistes_dev.db`, pull a balanced 29-asset
cross-asset universe, and verify: first/last rows, date range, null counts,
summary stats, and that there are **no missing dates** and **no forward-looking
leakage** (the next-day target must never be available at time t in the panel).

In [3]:

from core.db import get_storage
db = get_storage()

def load_universe(universe, min_bars=400):
    closes = {}
    for sym in universe:
        ohlcv = db.query_ohlcv(sym)
        if ohlcv is None or ohlcv.empty or len(ohlcv) < min_bars:
            print(f"  SKIP {sym}: insufficient data "
                  f"({0 if ohlcv is None else len(ohlcv)} bars)")
            continue
        closes[sym] = ohlcv["close"]
    prices = pd.DataFrame(closes).sort_index()
    prices = prices.dropna(axis=1, thresh=int(0.8 * len(prices)))
    return prices

prices = load_universe(UNIVERSE)
print(f"Loaded {prices.shape[1]} assets x {prices.shape[0]} bars")
print(f"Date range: {prices.index.min().date()} -> {prices.index.max().date()}")
prices.head()


Loaded 28 assets x 1306 bars
Date range: 2021-08-11 -> 2026-08-13


,AAPL,AMZN,GOOGL,JPM,META,MSFT,NVDA,TSLA,UNH,XOM,...,EEM,GLD,TLT,EURUSD=X,GBPUSD=X,USDJPY=X,USDINR=X,DX-Y.NYB,^TNX,GC=F
time,,,,,,,,,,,,,,,,,,,,,
2021-08-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.329,NaN
2021-08-12,145.223022,165.175003,135.988190,142.764496,359.493103,278.129730,19.836386,240.750000,371.861969,48.441319,...,46.109074,164.039993,122.907402,1.174190,1.386693,110.374001,74.102997,93.040001,1.367,1749.000000
2021-08-13,145.427872,164.698502,136.516983,141.167389,360.018433,281.047211,20.118412,239.056671,375.260376,47.951412,...,45.993492,166.389999,124.848785,1.173626,1.381158,110.403000,74.260101,92.519997,1.297,1775.199951
2021-08-16,147.398071,164.949493,137.093857,140.240860,363.369080,282.726624,19.881233,228.723328,382.166992,47.250351,...,45.575607,167.220001,125.151390,1.179468,1.386386,109.624001,74.216797,92.629997,1.257,1786.900024
2021-08-17,146.490982,162.098007,135.460342,138.546616,355.329651,281.267883,19.390926,221.903336,386.353149,46.929375,...,44.739841,166.970001,125.101006,1.177995,1.384275,109.296997,74.135002,93.129997,1.258,1785.000000


In [4]:

print("--- LAST 5 ROWS ---")
prices.tail()

# --- Null counts ---
nulls = prices.isna().sum()
print("--- Nulls per asset ---")
print(nulls[nulls > 0] if (nulls > 0).any() else "No nulls in the close panel.")

# --- Missing business days check (forward-looking safety) ---
mask = prices.notna().sum(axis=1)
all_present = prices[mask >= len(prices.columns) * 0.8]
idx = pd.to_datetime(all_present.index)
expected = pd.bdate_range(idx.min(), idx.max())
missing = expected.difference(idx)
print(f"Business days expected: {len(expected)} | present: {len(idx)} | missing: {len(missing)}")
if len(missing):
    print("Missing business days:", list(missing)[:10])


--- LAST 5 ROWS ---
--- Nulls per asset ---
AAPL           50
AMZN           51
GOOGL          51
JPM            51
META           51
MSFT           50
NVDA           51
TSLA           51
UNH            51
XOM            51
HDFCBANK.NS    66
INFY.NS        66
RELIANCE.NS    66
SBIN.NS        66
TCS.NS         66
SPY            50
QQQ            51
IWM            51
EEM            51
GLD            51
TLT            51
EURUSD=X        5
GBPUSD=X        5
USDJPY=X        5
USDINR=X        5
DX-Y.NYB       49
^TNX           50
GC=F           49
dtype: int64
Business days expected: 1305 | present: 1255 | missing: 50
Missing business days: [Timestamp('2021-09-06 00:00:00'), Timestamp('2021-11-25 00:00:00'), Timestamp('2021-12-24 00:00:00'), Timestamp('2022-01-17 00:00:00'), Timestamp('2022-02-21 00:00:00'), Timestamp('2022-04-15 00:00:00'), Timestamp('2022-05-30 00:00:00'), Timestamp('2022-06-20 00:00:00'), Timestamp('2022-07-04 00:00:00'), Timestamp('2022-09-05 00:00:00')]


In [5]:

# --- Summary statistics for a few representative assets ---
summary = pd.DataFrame({
    "mean_ret":  prices.pct_change().mean(),
    "std_ret":   prices.pct_change().std(),
    "annual_vol":prices.pct_change().std() * np.sqrt(252),
    "min_close": prices.min(),
    "max_close": prices.max(),
}).round(4)
summary.head(10)


,mean_ret,std_ret,annual_vol,min_close,max_close
AAPL,0.0010,0.0176,0.2792,122.8276,339.7869
AMZN,0.0007,0.0231,0.3668,81.8200,284.0200
GOOGL,0.0011,0.0203,0.3219,82.6967,402.3797
JPM,0.0009,0.0155,0.2454,93.3692,365.1800
META,0.0008,0.0285,0.4519,88.1360,787.4192
MSFT,0.0008,0.0179,0.2841,207.7340,538.6586
NVDA,0.0027,0.0328,0.5203,11.1992,235.4656
TSLA,0.0011,0.0374,0.5939,108.1000,489.8800
UNH,0.0003,0.0203,0.3224,231.5682,599.7756
XOM,0.0012,0.0168,0.2669,44.5390,170.3140


# Stage 2 — Feature Engineering & Target Construction

Compute standardized feature representations from the Oxford DL-for-finance
protocol:

- Volatility-normalized log returns at 1d / 5d / 21d / 63d / 126d:
  $$r^{\text{norm}}_{t,h} = \frac{r_{t,h}}{\sigma_t \sqrt{h}}$$
- Parkinson & Garman-Klass volatility estimators.
- Volatility-scaled next-day target:  $y_t = r_{t+1} / \sigma_t$

Print tensor dimensions `(N, Lookback, F)` and render a correlation heatmap.

In [6]:

from strategy_builder.features import build_features, build_target, FEATURE_COLS

# Build long-format panel with features + target, exactly like the benchmark.
from strategy_builder.features import build_universe_frame
panel = build_universe_frame(prices)
symbols = sorted(panel["symbol"].unique())
print(f"Panel: {len(panel):,} rows, {len(symbols)} symbols, "
      f"{panel['time'].min().date()} -> {panel['time'].max().date()}")
print("Feature columns:", FEATURE_COLS)
print("Target column  : target (vol-scaled next-day return, clipped)")
panel.head(3)


Panel: 28,201 rows, 28 symbols, 2022-08-01 -> 2026-08-13
Feature columns: ['ret_norm_1', 'ret_norm_5', 'ret_norm_21', 'ret_norm_63', 'ret_norm_126', 'ret_norm_252', 'macd_signal']
Target column  : target (vol-scaled next-day return, clipped)


,time,symbol,ret_1,sigma,vs_factor,target,ret_norm_1,ret_norm_5,ret_norm_21,ret_norm_63,ret_norm_126,ret_norm_252,macd_signal
6833,2022-08-01,EURUSD=X,0.001562,0.005421,184.468459,0.959707,0.288118,0.064838,-1.035125,-0.707001,-1.763995,-1.517785,-3.347643
6841,2022-08-01,USDINR=X,-0.004734,0.003125,320.048501,-0.966261,-1.515149,-1.222064,0.205023,1.438251,1.738336,1.385324,3.174786
6847,2022-08-01,GBPUSD=X,-0.000499,0.005797,172.495889,1.230234,-0.086064,1.161074,0.020151,-0.578988,-1.617173,-1.331785,-4.006837


In [7]:

# --- Tensor geometry (N, Lookback, F) for a single asset ---
from strategy_builder.trainer import WindowedDataset
sample_sym = panel["symbol"].iloc[0]
sub = panel[panel["symbol"] == sample_sym].sort_values("time")
ds = WindowedDataset(sub, FEATURE_COLS, lookback=LOOKBACK, symbols=[sample_sym])
print(f"Tensor shape: (N={ds.x.shape[0]}, Lookback={ds.x.shape[1]}, F={ds.x.shape[2]})")
print(f"Sample x[0]: {ds.x[0].shape}, next-day raw return r[0] = {ds.r[0]:.5f}")


Tensor shape: (N=1016, Lookback=32, F=7)
Sample x[0]: (32, 7), next-day raw return r[0] = 0.00020


In [8]:

# --- Feature correlation heatmap (representative asset) ---
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

feats = sub[FEATURE_COLS].dropna().corr()
fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(feats.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(feats))); ax.set_yticks(range(len(feats)))
ax.set_xticklabels(feats.columns, rotation=90); ax.set_yticklabels(feats.columns)
fig.colorbar(im, ax=ax, label="Pearson r")
ax.set_title(f"Feature correlation — {sample_sym}")
plt.tight_layout(); plt.show()


C:\Users\Priyatanshu Ghosh\AppData\Local\Temp\ipykernel_18904\2094363209.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


# Stage 3 — Walk-Forward Validation Engine (with Purge & Embargo)

Expanding-window walk-forward with a **strict purge gap** and **embargo**
between train and validation to eliminate lookahead bias: the validation
samples whose labels overlap the training horizon are purged, and an embargo
of `embargo_days` is dropped from the boundary.

> Without purge/embargo, overlapping labels leak information from the
> validation window back into training — inflating OOS metrics.

In [9]:

from datetime import timedelta

def walk_forward_purged(panel, train_months, test_months,
                        embargo_days=None, min_train=500, min_test=100):
    """Expanding-window splitter with purge + embargo.

    Returns list of (train, test) DataFrames. Embargo (in business days) drops
    a safety gap between the last training label and the first test bar.
    """
    start, end = panel["time"].min(), panel["time"].max()
    if embargo_days is None:
        embargo_days = max(5, int(test_months * 21 * 0.1))
    windows = []
    cursor = start + pd.DateOffset(months=train_months)
    step = pd.DateOffset(months=test_months)
    while cursor + step <= end:
        tr_end = cursor
        te_end = cursor + step
        tr = panel[(panel["time"] >= start) & (panel["time"] < tr_end)].copy()
        te = panel[(panel["time"] >= te_end - pd.Timedelta(days=embargo_days))
                   & (panel["time"] < te_end)].copy()
        # purge: drop test rows that overlap the last label window of train
        te = te[te["time"] >= tr_end + pd.Timedelta(days=embargo_days)]
        if len(tr) > min_train and len(te) > min_test:
            windows.append((tr, te))
        cursor += step
    return windows

WINDOWS = walk_forward_purged(panel, TRAIN_MONTHS, TEST_MONTHS,
                              embargo_days=max(5, int(TEST_MONTHS * 21 * 0.1)))
print(f"{len(WINDOWS)} walk-forward windows (train {TRAIN_MONTHS}m / "
      f"test {TEST_MONTHS}m, embargo purged)")

rows = []
for i, (tr, te) in enumerate(WINDOWS):
    rows.append({
        "window": i,
        "train_start": tr["time"].min().date(),
        "train_end":   tr["time"].max().date(),
        "train_rows":  len(tr),
        "test_start":  te["time"].min().date(),
        "test_end":    te["time"].max().date(),
        "test_rows":   len(te),
        "test_assets": te["symbol"].nunique(),
    })
wf_table = pd.DataFrame(rows)
display(wf_table) if "display" in dir() else print(wf_table.to_string(index=False))


6 walk-forward windows (train 24m / test 4m, embargo purged)
 window train_start  train_end  train_rows test_start   test_end  test_rows  test_assets
      0  2022-08-01 2024-07-31       13879 2024-11-25 2024-11-29        121           28
      1  2022-08-01 2024-11-29       16257 2025-03-24 2025-03-31        163           28
      2  2022-08-01 2025-03-31       18553 2025-07-24 2025-07-31        168           28
      3  2022-08-01 2025-07-31       20915 2025-11-24 2025-11-28        121           28
      4  2022-08-01 2025-11-28       23260 2026-03-24 2026-03-31        158           28
      5  2022-08-01 2026-03-31       25587 2026-07-24 2026-07-31        168           28


# Stage 4 — Model Training & Diagnostic Inspection

For each model family, fit on each walk-forward train slice and predict on the
OOS test slice:

1. **Neural sequence models** — NLinear, AR(1)x, DLinear, LSTM, PatchTST
   (PyTorch), trained on the pooled-Sharpe objective, **on the GPU pod**.
2. **Volatility regime models** — GARCH(1,1), EGARCH, GJR-GARCH, HAR-RV;
   inspect conditional variance.
3. **Instance & tree models** — k-NN, Lasso/ElasticNet, XGBoost, LightGBM,
   CatBoost.

Inspect raw predicted signals `s_t ∈ [-1,1]` and target position weights
`w_t = s_t · (σ_tgt / σ_t)`.

> In **smoke** mode this cell is trivially fast; in **full** mode on the pod it
> is the heavy GPU workload.

In [10]:

from strategy_builder.trainer import run_benchmark_model

def run_neural(name):
    t0 = time.time()
    res = run_benchmark_model(
        name, panel, FEATURE_COLS, symbols,
        lookback=LOOKBACK, hidden=HIDDEN, seeds=SEEDS, top_seeds=TOP_SEEDS,
        epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LR,
        train_months=TRAIN_MONTHS, test_months=TEST_MONTHS,
        sigma_tgt=SIGMA_TGT, use_ticker_emb=True, verbose=True, device=DEVICE)
    res["model"] = name
    res["seconds"] = round(time.time() - t0, 1)
    return res

NEURAL_RESULTS = {}
for name in NEURAL_MODELS:
    print(f"\n===== NEURAL: {name} =====")
    NEURAL_RESULTS[name] = run_neural(name)
    print(f"{name}: {len(NEURAL_RESULTS[name]['weights']):,} weight rows, "
          f"{NEURAL_RESULTS[name]['windows']} windows, "
          f"{NEURAL_RESULTS[name]['seconds']}s")
    # save per-model OOS weights
    NEURAL_RESULTS[name]["weights"].to_csv(BENCH_DIR / f"weights_{name}.csv", index=False)



===== NEURAL: nlinear =====


C:\Users\Priyatanshu Ghosh\Documents\Python Practice\CFA Practice\strategy_builder\trainer.py:160: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:823.)
  print(f"    {model_name} seed={seed} ep={ep} train_loss={float(loss):.3f} "


    nlinear seed=0 ep=0 train_loss=0.655 val_sharpe=0.448


    nlinear seed=0 ep=10 train_loss=-1.185 val_sharpe=0.558


    nlinear seed=0 ep=14 train_loss=-3.138 val_sharpe=0.303


...

    lstm seed=0 ep=0 train_loss=-1.676 val_sharpe=0.041


    lstm seed=0 ep=10 train_loss=-2.034 val_sharpe=0.505


    lstm seed=0 ep=14 train_loss=-1.458 val_sharpe=0.690


    lstm seed=1 ep=0 train_loss=-1.654 val_sharpe=0.297


lstm: 8,542 weight rows, 6 windows, 193.0s


In [11]:

# --- Inspect signals & weights for the first neural model ---
first_name = list(NEURAL_RESULTS.keys())[0]
w = NEURAL_RESULTS[first_name]["weights"]
print(f"Model: {first_name} | OOS weight rows: {len(w):,}")

# reconstruct position signal s_t in [-1,1] from weight = s * sigma_tgt / sigma
merged = w.merge(panel[["time", "symbol", "sigma"]], on=["time", "symbol"], how="left")
merged["signal"] = merged["weight"] / (SIGMA_TGT / merged["sigma"].clip(lower=1e-6))
print("\nPosition signal s_t:")
print(merged["signal"].describe().round(3))
print("\nTarget weight w_t:")
print(merged["weight"].describe().round(5))
print("\nSample of raw weights (time | symbol | weight):")
print(w.head(8).to_string(index=False))


Model: nlinear | OOS weight rows: 8,542

Position signal s_t:
count    8542.000
mean        0.279
std         0.602
min        -1.000
25%        -0.200
50%         0.428
75%         0.821
max         1.000
Name: signal, dtype: float64

Target weight w_t:
count    8542.00000
mean        2.54714
std         8.27161
min       -52.07692
25%        -1.40675
50%         2.88304
75%         5.82167
max        61.15401
Name: weight, dtype: float64

Sample of raw weights (time | symbol | weight):
      time      symbol    weight
2024-09-17 RELIANCE.NS  0.013550
2024-09-18 RELIANCE.NS  1.252858
2024-09-19 RELIANCE.NS  3.006489
2024-09-20 RELIANCE.NS -1.292920
2024-09-23 RELIANCE.NS -0.557702
2024-09-24 RELIANCE.NS  3.296055
2024-09-25 RELIANCE.NS  6.321820
2024-09-26 RELIANCE.NS  4.393946


In [12]:

# --- Volatility regime models: inspect conditional variance ---
import matplotlib
matplotlib.use("Agg")
from strategy_builder.volatility_models import (garch11_variance, egarch_variance,
                                                gjr_garch_variance, har_rv_forecast)

vol_sample = "SPY"
g = panel[panel["symbol"] == vol_sample].sort_values("time")
rets = g["ret_1"].fillna(0.0).to_numpy()

def series(cond_var):
    return pd.Series(cond_var, index=g["time"].values).dropna()

vols = {
    "GARCH(1,1)": series(garch11_variance(rets)),
    "EGARCH":     series(egarch_variance(rets)),
    "GJR-GARCH":  series(gjr_garch_variance(rets)),
    "EWMA(0.94)": series(np.fromiter(
        __import__("strategy_builder.volatility_models", fromlist=["ewma_variance"])
        .ewma_variance(rets), dtype=float)),
}
for name_, s in vols.items():
    print(f"{name_:<12s} mean σ² = {s.mean():.2e}  last σ = {np.sqrt(s.iloc[-1]):.4f}")

fig, ax = plt.subplots(figsize=(11, 4))
for name_, s in vols.items():
    ax.plot(s.index, np.sqrt(s).to_numpy(), label=name_, linewidth=1)
ax.set_title(f"Conditional volatility σ_t — {vol_sample}")
ax.set_ylabel("σ_t"); ax.legend(ncol=4); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


GARCH(1,1)   mean σ² = 1.05e-04  last σ = 0.0093
EGARCH       mean σ² = 1.05e-04  last σ = 0.0093
GJR-GARCH    mean σ² = 1.05e-04  last σ = 0.0093
EWMA(0.94)   mean σ² = 1.09e-04  last σ = 0.0084


C:\Users\Priyatanshu Ghosh\AppData\Local\Temp\ipykernel_18904\3456539928.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [13]:

# --- Classical / tree / instance models ---
from strategy_builder.classical import CLASSICAL_REGISTRY

CLASSICAL_RESULTS = {}
for name in CLASSICAL_MODELS:
    if name not in CLASSICAL_REGISTRY:
        print(f"SKIP {name}: not in registry")
        continue
    t0 = time.time()
    try:
        weights = CLASSICAL_REGISTRY[name](
            panel, FEATURE_COLS, symbols,
            lookback=LOOKBACK, train_months=TRAIN_MONTHS, test_months=TEST_MONTHS)
        CLASSICAL_RESULTS[name] = {"model": name, "weights": weights,
                                   "seconds": round(time.time() - t0, 1)}
        weights.to_csv(BENCH_DIR / f"weights_{name}.csv", index=False)
        print(f"{name}: {len(weights):,} weight rows, "
              f"{CLASSICAL_RESULTS[name]['seconds']}s")
    except Exception as e:  # noqa: BLE001
        import traceback; traceback.print_exc()
        CLASSICAL_RESULTS[name] = {"model": name, "error": str(e),
                                   "weights": pd.DataFrame(columns=["time","symbol","weight"])}


ridge: 8,542 weight rows, 4.6s


xgboost: 8,542 weight rows, 16.2s


# Stage 5 — Rigorous Financial Metric & Statistical Verification

For every model's combined OOS portfolio return series compute:

- Annualized **Sharpe** & **Sortino** ratio
- **CAGR %** & **Max Drawdown %**
- **Calmar** & **Win Rate (Hit %)**
- **Newey-West HAC** t-statistic (H₀: zero alpha)
- **Breakeven Transaction Friction** $c^*$ (bps)

Render an interactive cumulative equity curve (models vs Buy-and-Hold).

In [14]:

from strategy_builder.backtest import (full_metrics, passive_benchmark,
                                       breakeven_costs, equity_curve_from_weights)

passive = passive_benchmark(panel)
print("Passive (equal-weight long-only) daily returns:", len(passive))

ALL_WEIGHTS = {}
for d in list(NEURAL_RESULTS.values()) + list(CLASSICAL_RESULTS.values()):
    if d.get("weights") is not None and not d["weights"].empty:
        ALL_WEIGHTS[d["model"]] = d["weights"]

summary = []
for name, w in ALL_WEIGHTS.items():
    try:
        m = full_metrics(w, panel, passive)
        m["model"] = name
        summary.append(m)
        print(f"{name:<12s} sharpe={m['sharpe']:+.3f} cagr={m['cagr']*100:+.1f}% "
              f"maxDD={m['max_dd']*100:.1f}% calmar={m['calmar']:.2f} "
              f"hit={m['hit_rate']*100:.1f}% t_hac={m['t_hac']:+.2f} "
              f"c*={m.get('breakeven_bps', 0):.1f}bps")
    except Exception as e:  # noqa: BLE001
        print(f"{name}: METRIC FAIL {e}")

summary.sort(key=lambda s: s.get("sharpe", -99), reverse=True)
leaderboard = pd.DataFrame(summary)
leaderboard_path = BENCH_DIR / "leaderboard.json"
leaderboard_path.write_text(
    json.dumps({"mode": RUN_MODE, "generated": pd.Timestamp.now().isoformat(),
                "summary": summary}, indent=2, default=str),
    encoding="utf-8")
print("\nLeaderboard saved ->", leaderboard_path)
print("\n===== LEADERBOARD (sorted by OOS Sharpe) =====")
cols = ["model","sharpe","cagr","max_dd","calmar","hit_rate","t_hac","turnover"]
print(leaderboard[cols].to_string(index=False))


Passive (equal-weight long-only) daily returns: 1053
nlinear      sharpe=+1.183 cagr=+44.0% maxDD=-39.1% calmar=1.13 hit=57.8% t_hac=+1.25 c*=0.0bps
lstm         sharpe=-3.790 cagr=-17.7% maxDD=-21.7% calmar=-0.81 hit=40.4% t_hac=-3.96 c*=0.0bps


ridge        sharpe=-3.961 cagr=-0.0% maxDD=-0.0% calmar=-0.73 hit=43.2% t_hac=-3.05 c*=0.0bps
xgboost      sharpe=-0.783 cagr=-0.0% maxDD=-0.0% calmar=-0.29 hit=48.8% t_hac=-0.90 c*=0.0bps

Leaderboard saved -> data\benchmark\leaderboard.json

===== LEADERBOARD (sorted by OOS Sharpe) =====
  model  sharpe    cagr  max_dd  calmar  hit_rate  t_hac  turnover
nlinear   1.183  0.4404 -0.3909   1.127     0.578   1.25   35264.9
xgboost  -0.783 -0.0000 -0.0001  -0.293     0.488  -0.90       3.1
   lstm  -3.790 -0.1771 -0.2173  -0.815     0.404  -3.96    2010.9
  ridge  -3.961 -0.0000 -0.0001  -0.726     0.432  -3.05       0.3


In [15]:

# --- Cumulative equity curves vs Buy-and-Hold passive ---
import matplotlib
matplotlib.use("Agg")
fig, ax = plt.subplots(figsize=(12, 6))
bench = (1 + passive).cumprod()
ax.plot(bench.index, bench.to_numpy(), color="black", linewidth=2,
        label="Buy & Hold (passive)")

for name, w in ALL_WEIGHTS.items():
    try:
        ec = equity_curve_from_weights(w, panel)
        ax.plot(ec.index, ec.to_numpy(), linewidth=1, label=name)
    except Exception as e:  # noqa: BLE001
        print(f"{name}: curve fail {e}")
ax.set_yscale("log")
ax.set_title("Cumulative equity — all models vs Buy & Hold")
ax.set_ylabel("Equity (log)"); ax.legend(ncol=4, fontsize=8)
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


C:\Users\Priyatanshu Ghosh\AppData\Local\Temp\ipykernel_18904\3566245302.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


In [16]:

# --- Breakeven transaction friction c* (bps) for top models ---
be = {}
top = summary[: min(4, len(summary))]
for m in top:
    name = m["model"]
    try:
        be[name] = breakeven_costs(ALL_WEIGHTS[name], panel).head(8).to_dict("records")
    except Exception as e:  # noqa: BLE001
        be[name] = {"error": str(e)}
(BENCH_DIR / "breakeven.json").write_text(json.dumps(be, indent=2, default=str),
                                          encoding="utf-8")
for name, recs in be.items():
    print(f"\n{name} — top-8 breakeven costs (bps):")
    if isinstance(recs, list):
        for r in recs:
            print(f"  {r.get('symbol','?'):<12s} c* = {r.get('breakeven_bps',0):>8.1f} bps "
                  f"(turnover/yr {r.get('turnover_ann',0):.1f})")
    else:
        print(" ", recs)



nlinear — top-8 breakeven costs (bps):
  JPM          c* =     22.0 bps (turnover/yr 716.8)
  XOM          c* =     17.5 bps (turnover/yr 908.9)
  EEM          c* =     16.6 bps (turnover/yr 1109.1)
  SBIN.NS      c* =     14.5 bps (turnover/yr 790.1)
  META         c* =     14.2 bps (turnover/yr 478.7)
  IWM          c* =     14.0 bps (turnover/yr 953.7)
  GC=F         c* =     13.7 bps (turnover/yr 1034.4)
  GLD          c* =     13.2 bps (turnover/yr 1000.1)

xgboost — top-8 breakeven costs (bps):
  QQQ          c* =     26.4 bps (turnover/yr 0.1)
  NVDA         c* =     16.1 bps (turnover/yr 0.0)
  JPM          c* =      6.9 bps (turnover/yr 0.0)
  EURUSD=X     c* =      4.6 bps (turnover/yr 0.3)
  GBPUSD=X     c* =      4.5 bps (turnover/yr 0.2)
  SPY          c* =      4.0 bps (turnover/yr 0.1)
  AMZN         c* =      4.0 bps (turnover/yr 0.0)
  USDINR=X     c* =      3.9 bps (turnover/yr 0.4)

lstm — top-8 breakeven costs (bps):
  AAPL         c* =     61.4 bps (turnover/yr 38

# Stage 6 — Modular Extensibility (Paper Replication Template)

A standardized `@register_model` decorator so any newly published arXiv/NeurIPS
trading paper or custom alpha formula can be added in a single cell and
**automatically** evaluated against the master leaderboard.

New models only need to implement the standard interface:
`fn(panel, feature_cols, symbols, **kw) -> DataFrame[time, symbol, weight]`.

In [17]:

# --- Model registry with @register_model ---
import functools

MODEL_REGISTRY = {}

def register_model(name=None, family="custom", description=""):
    """Decorator: register a model factory for the master leaderboard.

    The wrapped callable must return DataFrame[time, symbol, weight].
    """
    def deco(fn):
        key = name or fn.__name__
        MODEL_REGISTRY[key] = {
            "fn": fn, "family": family, "description": description,
        }
        return fn
    return deco

@register_model("momentum_cross", family="alpha",
                description="21d vs 63d EWMA crossover, vol-scaled")
def momentum_cross(panel, feature_cols, symbols, lookback=64,
                   train_months=36, test_months=6, sigma_tgt=0.10):
    rows = []
    for sym, g in panel.groupby("symbol", sort=False):
        g = g.sort_values("time")
        fast = g["close"] if "close" in g else np.exp(np.log1p(g["ret_1"]).cumsum())
        e21 = fast.ewm(span=21, adjust=False).mean()
        e63 = fast.ewm(span=63, adjust=False).mean()
        sig = np.sign(e21 - e63)
        for i, (_, r) in enumerate(g.iterrows()):
            vs = 1.0 / max(float(r.get("sigma", 1e-4)), 1e-6)
            rows.append({"time": r["time"], "symbol": sym,
                         "weight": float(sig.iloc[i] * sigma_tgt * vs)})
    return pd.DataFrame(rows)

print("Registered models:", sorted(MODEL_REGISTRY))


Registered models: ['momentum_cross']


In [18]:

# --- Evaluate every registered custom model on the master leaderboard ---
for key, spec in MODEL_REGISTRY.items():
    t0 = time.time()
    try:
        w = spec["fn"](panel, FEATURE_COLS, symbols, lookback=LOOKBACK,
                       train_months=TRAIN_MONTHS, test_months=TEST_MONTHS)
        w.to_csv(BENCH_DIR / f"weights_{key}.csv", index=False)
        m = full_metrics(w, panel, passive)
        m["model"] = key; m["family"] = spec["family"]
        summary.append(m)
        print(f"[{spec['family']:>5s}] {key:<20s} sharpe={m['sharpe']:+.3f} "
              f"cagr={m['cagr']*100:+.1f}% maxDD={m['max_dd']*100:.1f}% "
              f"t_hac={m['t_hac']:+.2f} ({round(time.time()-t0,1)}s)")
    except Exception as e:  # noqa: BLE001
        import traceback; traceback.print_exc()
        print(f"[{key}] FAILED: {e}")

# Rewrite leaderboard to include custom models
summary.sort(key=lambda s: s.get("sharpe", -99), reverse=True)
leaderboard = pd.DataFrame(summary)
(BENCH_DIR / "leaderboard.json").write_text(
    json.dumps({"mode": RUN_MODE, "generated": pd.Timestamp.now().isoformat(),
                "summary": summary}, indent=2, default=str), encoding="utf-8")
print("\nFinal leaderboard:")
cols = ["model","family","sharpe","cagr","max_dd","calmar","hit_rate","t_hac"]
print(leaderboard[[c for c in cols if c in leaderboard.columns]].to_string(index=False))


[alpha] momentum_cross       sharpe=+1.528 cagr=+97.8% maxDD=-41.3% t_hac=+3.14 (1.3s)

Final leaderboard:
         model family  sharpe    cagr  max_dd  calmar  hit_rate  t_hac
momentum_cross  alpha   1.528  0.9778 -0.4134   2.365     0.557   3.14
       nlinear    NaN   1.183  0.4404 -0.3909   1.127     0.578   1.25
       xgboost    NaN  -0.783 -0.0000 -0.0001  -0.293     0.488  -0.90
          lstm    NaN  -3.790 -0.1771 -0.2173  -0.815     0.404  -3.96
         ridge    NaN  -3.961 -0.0000 -0.0001  -0.726     0.432  -3.05


## End of 6-stage empirical research lab

- **Stage 1** — verified clean, leakage-free 29-asset cross-asset panel.
- **Stage 2** — standardized vol-normalized features + target; tensor `(N, L, F)`.
- **Stage 3** — purge & embargo walk-forward splits.
- **Stage 4** — neural / volatility / tree models trained (GPU on pod).
- **Stage 5** — full metric suite + equity curves + breakeven costs.
- **Stage 6** — `@register_model` paper-replication template.

Artifacts: `data/benchmark/weights_*.csv`, `data/benchmark/leaderboard.json`,
`data/benchmark/breakeven.json`.

To add a new paper model, define it in a cell with `@register_model`, then
re-run Stage 6 — it is scored automatically on the master leaderboard.

# Stage 7 — Research Results Viewer

Loads the committed research artifacts from `research/results/` and renders them as tables + charts so every result can be observed directly here.

In [19]:

# ==== STAGE 7 : RESEARCH RESULTS VIEWER ====
import json
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RESULTS = Path("research/results")
print("Artifacts in research/results:")
for p in sorted(RESULTS.glob("*")):
    print(f"  {p.name:<45} {p.stat().st_size:>9,} B")


Artifacts in research/results:
  01_ma_crossover_results.csv                      14,897 B
  01_ma_crossover_results.json                     63,542 B
  weights_catboost.csv                            134,318 B
  weights_lightgbm.csv                            134,706 B
  weights_logistic.csv                            123,244 B
  weights_ridge.csv                               134,191 B
  weights_xgboost.csv                             134,634 B


### 7.1 MA-Crossover strategy backtests

Per-strategy × per-symbol results: total return, CAGR, Sharpe, Sortino, Calmar, max drawdown, win rate, profit factor, alpha vs SPY.

In [20]:

# ==== STAGE 7 : RESEARCH RESULTS VIEWER ====
csv_path = RESULTS / "01_ma_crossover_results.csv"
if csv_path.exists():
    ma = pd.read_csv(csv_path)
    print(f"{len(ma)} backtests across {ma['strategy'].nunique()} strategies "
          f"x {ma['symbol'].nunique()} symbols\n")
    show = ["strategy", "symbol", "total_return_pct", "cagr_pct", "sharpe",
            "sortino", "calmar", "max_drawdown_pct", "win_rate", "profit_factor",
            "alpha_vs_spy_pct", "info_ratio"]
    display(ma[[c for c in show if c in ma.columns]].head(25))
else:
    print("01_ma_crossover_results.csv not found")


80 backtests across 4 strategies x 20 symbols



,strategy,symbol,total_return_pct,cagr_pct,sharpe,sortino,calmar,max_drawdown_pct,win_rate,profit_factor,alpha_vs_spy_pct,info_ratio
0,ma_cross_20_100,AAPL,27.72,5.01,0.765,1.171,0.767,-6.54,0.619,3.964,-8.91,-0.479
1,ma_cross_20_100,MSFT,30.20,5.42,0.417,0.886,0.719,-7.53,0.571,2.630,-7.79,-0.339
2,ma_cross_20_100,NVDA,327.20,33.70,0.450,5.320,1.404,-24.00,0.600,11.618,52.75,0.353
3,ma_cross_20_100,AMZN,22.06,4.07,0.382,0.395,0.151,-26.92,0.609,1.767,-9.16,-0.436
4,ma_cross_20_100,GOOGL,59.69,9.81,0.570,0.989,0.509,-19.30,0.632,3.399,-2.87,-0.111
5,ma_cross_20_100,META,125.69,17.68,0.381,1.519,0.538,-32.85,0.500,3.805,26.77,0.247
6,ma_cross_20_100,TSLA,33.19,5.90,0.436,0.443,0.198,-29.83,0.545,1.394,-6.93,-0.297
7,ma_cross_20_100,JPM,42.04,7.27,0.508,1.008,0.558,-13.02,0.706,3.871,-5.76,-0.247
8,ma_cross_20_100,RELIANCE.NS,14.39,2.72,0.546,0.639,0.927,-2.94,0.550,1.822,0.00,0.000
9,ma_cross_20_100,HDFCBANK.NS,10.32,1.98,0.363,0.283,0.253,-7.83,0.667,1.079,0.00,0.000


In [21]:

# ==== STAGE 7 : RESEARCH RESULTS VIEWER ====
json_path = RESULTS / "01_ma_crossover_results.json"
if json_path.exists():
    d = json.load(open(json_path, encoding="utf-8"))
    avg = d.get("per_strategy_avg", {})
    rows = [{"strategy": k, **v} for k, v in avg.items()]
    avg_df = pd.DataFrame(rows)
    print("Per-strategy averages (across symbols):")
    display(avg_df.sort_values("avg_sharpe", ascending=False))
else:
    print("01_ma_crossover_results.json not found")


Per-strategy averages (across symbols):


,strategy,avg_return_pct,avg_sharpe,avg_max_dd_pct,avg_win_rate,n_tested
0,ma_cross_20_100,44.60,0.276,-17.36,0.554,20
3,ma_cross_50_200,32.25,0.253,-11.17,0.561,20
1,ma_cross_20_100_short,185.90,0.250,-58.58,0.563,20
2,ma_cross_10_50,28.00,0.217,-22.21,0.534,20


In [22]:

# ==== STAGE 7 : RESEARCH RESULTS VIEWER ====
# Sharpe / return / drawdown by strategy (average across symbols)
if "avg_df" in dir():
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    x = avg_df["strategy"]
    for ax, col, title in [
        (axes[0], "avg_sharpe", "Avg Sharpe"),
        (axes[1], "avg_return_pct", "Avg Return %"),
        (axes[2], "avg_max_dd_pct", "Avg Max Drawdown %"),
    ]:
        ax.bar(x, avg_df[col], color="#4f8cff")
        ax.set_title(title); ax.set_xlabel("Strategy"); ax.tick_params(axis="x", rotation=20)
    fig.tight_layout(); plt.show()


C:\Users\Priyatanshu Ghosh\AppData\Local\Temp\ipykernel_18904\1166621994.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


### 7.2 Cross-sectional model weights (strategy_builder benchmark)

The committed `weights_*.csv` are each model's per-symbol allocation over time from the benchmark. Plot the top exposures and turnover.

In [23]:

# ==== STAGE 7 : RESEARCH RESULTS VIEWER ====
import glob as _glob
weight_files = sorted(_glob.glob(str(RESULTS / "weights_*.csv")))
print(f"Found {len(weight_files)} weight files: {[Path(f).name for f in weight_files]}")
weight_frames = {}
for f in weight_files:
    name = Path(f).stem.replace("weights_", "")
    weight_frames[name] = pd.read_csv(f, parse_dates=["time"])
    print(f"  {name:<12} rows={len(weight_frames[name]):,}  cols={list(weight_frames[name].columns)}")


Found 5 weight files: ['weights_catboost.csv', 'weights_lightgbm.csv', 'weights_logistic.csv', 'weights_ridge.csv', 'weights_xgboost.csv']
  catboost     rows=3,289  cols=['time', 'symbol', 'weight']
  lightgbm     rows=3,289  cols=['time', 'symbol', 'weight']
  logistic     rows=3,289  cols=['time', 'symbol', 'weight']
  ridge        rows=3,289  cols=['time', 'symbol', 'weight']
  xgboost      rows=3,289  cols=['time', 'symbol', 'weight']


In [24]:

# ==== STAGE 7 : RESEARCH RESULTS VIEWER ====
# Turnover + top exposures per model (avg |dw| per rebalance)
if weight_frames:
    rows = []
    for name, w in weight_frames.items():
        w = w.sort_values("time")
        # daily turnover = sum of |weight change| / 2
        piv = w.pivot_table(index="time", columns="symbol", values="weight").fillna(0)
        if len(piv) > 1:
            turnover = (piv.diff().abs().sum(axis=1) / 2).mean()
        else:
            turnover = float("nan")
        rows.append({"model": name, "rows": len(w), "avg_daily_turnover": turnover})
    wt = pd.DataFrame(rows)
    print("Daily turnover per model (avg across the OOS window):")
    display(wt)


Daily turnover per model (avg across the OOS window):


,model,rows,avg_daily_turnover
0,catboost,3289,0.002536
1,lightgbm,3289,0.005330
2,logistic,3289,1.777630
3,ridge,3289,0.000749
4,xgboost,3289,0.005128


In [25]:

# ==== STAGE 7 : RESEARCH RESULTS VIEWER ====
# Top-N exposures over time for the best available model (first with data)
if weight_frames:
    name = next(iter(weight_frames))
    w = weight_frames[name].sort_values("time")
    piv = w.pivot_table(index="time", columns="symbol", values="weight").fillna(0)
    fig, ax = plt.subplots(figsize=(14, 4))
    # Signed weights (long/short) can't be stacked — use plain lines.
    piv.plot(ax=ax, alpha=0.6, legend=False)
    ax.set_title(f"Exposures over time — {name}")
    ax.set_ylabel("weight"); ax.set_xlabel("time")
    # legend: top 6 symbols by mean weight
    top = piv.mean().sort_values(ascending=False).head(6).index
    ax.legend(top, loc="upper right", fontsize=8, ncol=3)
    fig.tight_layout(); plt.show()


C:\Users\Priyatanshu Ghosh\AppData\Local\Temp\ipykernel_18904\2266266249.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()
